# 23b_evaluate_ensemble — 표현앙상블 모델 평가 6종 (학습곡선 CV 최신화)

**한 줄 요약:** 배포 모델을 6가지로 검증. **panel 1 학습곡선은 5-fold CV + 음영(±std)** 으로 최신화(단일 측정 → CV 평균).
**나머지 5종:** ROC/AUC·Confusion·Prediction Error·Calibration·Feature Importance (test셋 기준).
**출력:** `data/ensemble_evaluation.png`.

### 전체 평가 실행 (학습곡선=CV 음영)

In [ ]:
import os
while not os.path.isdir('data') and os.path.dirname(os.getcwd()) != os.getcwd():
    os.chdir('..')
print('작업 폴더:', os.getcwd())

import os, numpy as np, pandas as pd, pickle
from collections import defaultdict
from sklearn.base import clone
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (roc_curve, roc_auc_score, confusion_matrix,
                             matthews_corrcoef, accuracy_score)
from sklearn.calibration import calibration_curve
import matplotlib; matplotlib.use("Agg"); import matplotlib.pyplot as plt

with open("data/HSD17B13_repr_ensemble.pkl","rb") as f: B=pickle.load(f)
TOP, REPS, PIPES = B["top"], B["reps"], B["pipelines"]
feat = pd.read_csv("data/HSD17B13_final_training_1to1_v2.csv")
mem  = pd.read_csv("data/HSD17B13_rebalanced_membership.csv")
df = mem.merge(feat.drop(columns=["potency"]), on="canonical_smiles", how="left")

def Xy(rep, split):
    m = df["split"]==split
    return df.loc[m, REPS[rep]].to_numpy(), df.loc[m,"potency"].to_numpy(), df.loc[m,"source"].to_numpy()
def ens_proba(split):
    return np.mean([PIPES[f"{m}|{r}"].predict_proba(df.loc[df.split==split,REPS[r]].to_numpy())[:,1] for m,r in TOP],0)

yte = df.loc[df.split=="test","potency"].to_numpy(); src_te = df.loc[df.split=="test","source"].to_numpy()
p_te = ens_proba("test"); pred_te=(p_te>=0.5).astype(int)
print("test MCC %.3f | ROC-AUC %.3f | Acc %.3f" % (matthews_corrcoef(yte,pred_te), roc_auc_score(yte,p_te), accuracy_score(yte,pred_te)))

fig, ax = plt.subplots(3,2, figsize=(13,15)); fig.suptitle("HSD17B13 Representation-Ensemble — Evaluation", fontsize=15, y=0.995)

# 1) Learning Curve — CV(5-fold) + std 음영 (accuracy)
dev = df[df.split!="test"].reset_index(drop=True)
used=sorted(set(r for _,r in TOP)); col_order=[]; ranges={}
for r in used:
    s=len(col_order); col_order+=REPS[r]; ranges[r]=(s,len(col_order))
Xd=dev[col_order].to_numpy(); yd=dev["potency"].to_numpy()
specs=[(ranges[rn], clone(PIPES[f"{mn}|{rn}"])) for mn,rn in TOP]
def fit_ens(Xtr,ytr): return [((a,b),clone(t).fit(Xtr[:,a:b],ytr)) for (a,b),t in specs]
def pr(fit,Xx): return np.mean([p.predict_proba(Xx[:,a:b])[:,1] for (a,b),p in fit],0)
fracs=[0.2,0.4,0.6,0.8,1.0]; skf=StratifiedKFold(5,shuffle=True,random_state=42); rng=np.random.RandomState(0)
TR,VA=[],[]
for f in fracs:
    a_tr,a_va=[],[]
    for tri,vai in skf.split(Xd,yd):
        tri=tri.copy(); rng.shuffle(tri); sub=tri[:max(2,int(len(tri)*f))]
        fit=fit_ens(Xd[sub],yd[sub])
        a_tr.append(accuracy_score(yd[sub],(pr(fit,Xd[sub])>=.5))); a_va.append(accuracy_score(yd[vai],(pr(fit,Xd[vai])>=.5)))
    TR.append(a_tr); VA.append(a_va)
TR=np.array(TR); VA=np.array(VA); ns=[int(len(Xd)*0.8*f) for f in fracs]
ax[0,0].plot(ns,TR.mean(1),"o-",color="tab:blue",label="Training score")
ax[0,0].fill_between(ns,TR.mean(1)-TR.std(1),TR.mean(1)+TR.std(1),alpha=.2,color="tab:blue")
ax[0,0].plot(ns,VA.mean(1),"s-",color="green",label="CV score")
ax[0,0].fill_between(ns,VA.mean(1)-VA.std(1),VA.mean(1)+VA.std(1),alpha=.2,color="green")
ax[0,0].set_title("1) Learning Curve (5-fold CV, +/-std)"); ax[0,0].set_xlabel("Training samples"); ax[0,0].set_ylabel("Accuracy")
ax[0,0].legend(); ax[0,0].grid(alpha=.3)
print("Learning curve CV: val acc %.3f -> %.3f" % (VA.mean(1)[0], VA.mean(1)[-1]))

# 2) ROC
fpr,tpr,_=roc_curve(yte,p_te); auc=roc_auc_score(yte,p_te)
ax[0,1].plot(fpr,tpr,label=f"AUC = {auc:.3f}"); ax[0,1].plot([0,1],[0,1],"k--",alpha=.5)
ax[0,1].set_title("2) ROC Curve"); ax[0,1].set_xlabel("False Positive Rate"); ax[0,1].set_ylabel("True Positive Rate"); ax[0,1].legend(loc="lower right"); ax[0,1].grid(alpha=.3)
# 3) Confusion
cm=confusion_matrix(yte,pred_te,labels=[0,1]); ax[1,0].imshow(cm,cmap="Greens")
for (i,j),v in np.ndenumerate(cm): ax[1,0].text(j,i,str(v),ha="center",va="center",color="white" if v>cm.max()/2 else "black",fontsize=14)
ax[1,0].set_xticks([0,1]);ax[1,0].set_xticklabels(["inactive(0)","active(1)"]);ax[1,0].set_yticks([0,1]);ax[1,0].set_yticklabels(["inactive(0)","active(1)"])
ax[1,0].set_title("3) Confusion Matrix"); ax[1,0].set_xlabel("Predicted"); ax[1,0].set_ylabel("Actual")
# 4) Prediction error by source
groups=["active","decoy","real_inactive"]
p1=np.array([(pred_te[src_te==g]==1).sum() for g in groups]); p0=np.array([(pred_te[src_te==g]==0).sum() for g in groups])
ax[1,1].bar(groups,p0,label="pred inactive(0)",color="#8ecae6"); ax[1,1].bar(groups,p1,bottom=p0,label="pred active(1)",color="#fb8500")
for i,g in enumerate(groups): ax[1,1].text(i,(p0[i]+p1[i])+1,f"n={p0[i]+p1[i]}",ha="center",fontsize=9)
ax[1,1].set_title("4) Class Prediction Error (by source)"); ax[1,1].set_ylabel("count"); ax[1,1].legend()
# 5) Calibration
frac,mpv=calibration_curve(yte,p_te,n_bins=10,strategy="uniform")
ax[2,0].plot(mpv,frac,"o-",label="Ensemble"); ax[2,0].plot([0,1],[0,1],"k--",alpha=.5,label="Perfect")
ax[2,0].set_title("5) Calibration Curve"); ax[2,0].set_xlabel("Mean predicted prob"); ax[2,0].set_ylabel("Fraction of positives"); ax[2,0].legend(); ax[2,0].grid(alpha=.3)
# 6) Feature importance
imp=defaultdict(float)
for mn,rn in TOP:
    model=PIPES[f"{mn}|{rn}"].named_steps["m"]; fi=getattr(model,"feature_importances_",None)
    if fi is None: continue
    for c,val in zip(REPS[rn],fi):
        name=c if rn=="desc2d" else f"{rn}:{c.split('_')[-1]}"; imp[name]+=val/len(TOP)
topf=sorted(imp.items(),key=lambda x:x[1],reverse=True)[:15][::-1]
ax[2,1].barh([k for k,_ in topf],[v for _,v in topf],color="#219ebc")
ax[2,1].set_title("6) Feature Importance (ensemble aggregate, top15)"); ax[2,1].set_xlabel("mean importance"); ax[2,1].tick_params(axis="y",labelsize=8)

plt.tight_layout(rect=[0,0,1,0.99]); plt.savefig("data/ensemble_evaluation.png",dpi=130,bbox_inches="tight")
print("저장(최신화): data/ensemble_evaluation.png")
